
# ⚙️ Klike Recommendations Engine

Este notebook apresenta o protótipo interativo do sistema de recomendações desenvolvido na Parte 3 do Klike Data Science Challenge.

O sistema analisa uma campanha existente e gera até cinco sugestões de melhoria do criativo, considerando suas características, a plataforma de veiculação e os padrões de performance observados no dataset.

As recomendações são específicas, quantificadas e apresentadas em linguagem natural, com o objetivo de facilitar sua interpretação por profissionais de marketing.

A metodologia de construção, os critérios de priorização e os testes estão documentados no notebook `parte3_recomendacoes.ipynb`.



## 1. Preparação dos dados

O protótipo utiliza as funções disponíveis em `src/recommendations.py`, evitando duplicar a implementação do Recommendations Engine.

Nesta etapa, o dataset original é carregado e preparado pela função `preparar_dados()`, responsável por criar as variáveis derivadas necessárias às comparações, incluindo grupos de duração e áudio, taxa de conversão e proporção assistida do vídeo.

A preparação é realizada uma única vez, antes da interação com o usuário. As campanhas são posteriormente selecionadas a partir desse conjunto de dados preparado.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import pandas as pd
import io
import html
from contextlib import redirect_stdout

raiz_projeto = Path.cwd().resolve().parent

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.recommendations import (preparar_dados, recommendations_engine, apresentar_recomendacoes)

dados = pd.read_csv(raiz_projeto / "data" / "klike_challenge_dataset.csv")
dados = preparar_dados(dados)


## 2. Demonstração interativa

A interface permite selecionar uma campanha existente no dataset e visualizar as recomendações geradas pelo sistema.

**Como utilizar:**

1. Execute a célula de preparação dos dados.
2. Informe o número da campanha no campo indicado. Por exemplo, digite `90` para selecionar `KLK-0090`.
3. Clique em **Gerar recomendações**.
4. Analise as sugestões apresentadas abaixo do formulário.

Para cada campanha, o sistema apresenta:

- Plataforma e objetivo da campanha.
- Até cinco recomendações de alteração do criativo.
- Diferenças percentuais observadas nas métricas de performance.

As recomendações são selecionadas com base em comparações entre grupos da mesma plataforma. O sistema considera diferenças entre medianas de pelo menos 10% e exige um mínimo de 10 observações válidas por grupo.

**Observação:** o sistema pode retornar menos de cinco recomendações quando não existem alternativas que atendam aos critérios definidos.


In [3]:
import ipywidgets as widgets
from IPython.display import display

# Campo para inserir o número da campanha
campo_numero = widgets.Text(value="", placeholder="Ex.: 0090", layout=widgets.Layout(width="250px"))
prefixo = widgets.HTML(value="<span style='font-size: 15px;'>KLK-</span>")

# Botão para executar o engine
botao_analisar = widgets.Button(description="Gerar recomendações", style={"button_color": "#8ad978", "font_weight": "600"}, 
                                icon="lightbulb-o", layout=widgets.Layout(width="200px"))

# Área onde os resultados serão exibidos
saida = widgets.HTML(value="")

def analisar_campanha(_):
    numero = campo_numero.value.strip()

    if not numero.isdigit():
        saida.value = "<p>Informe apenas o número da campanha.</p>"
        return

    campanha_id = f"KLK-{numero.zfill(4)}"
    campanhas = dados.loc[dados["campaign_id"] == campanha_id]

    if campanhas.empty:
        saida.value = f"<p>Campanha {campanha_id} não encontrada.</p>"
        return

    campanha = campanhas.iloc[0]

    try:
        resultado = recommendations_engine(campanha, dados)

        buffer = io.StringIO()

        with redirect_stdout(buffer):
            apresentar_recomendacoes(campanha, resultado)

        texto = html.escape(buffer.getvalue())

        saida.value = ("<pre style='white-space: pre-wrap; " "font-size: 13px; line-height: 1.6;'>" f"{texto}</pre>")

    except Exception as erro:
        saida.value = (f"<p>Erro ao analisar a campanha: " f"{html.escape(str(erro))}</p>")

titulo = widgets.HTML(
    """<div style="background-color: #EAF0FF; border-left: 5px solid #f6df5a; border-radius: 8px; padding: 16px 20px;">
        <h2 style="color: #5275DA; margin: 0 0 12px 0; font-size: 24px;">⚙️ Klike Recommendations Engine</h2>
        <p style="color: #475569; margin: 0; font-size: 15px;">Informe o número de uma campanha para receber sugestões de melhoria do criativo.</p>
       </div>
    """)

botao_analisar.on_click(analisar_campanha)

# Exibe a interface
interface = widgets.VBox(
    [titulo,
    widgets.HBox(
        [prefixo, campo_numero, botao_analisar],
        layout=widgets.Layout(align_items="center", gap="8px")),
    saida],
    layout=widgets.Layout(border="1px solid #E5E7EB", padding="16px", width="100%"))


display(interface)


## 3. Interpretação dos resultados

As diferenças percentuais apresentadas representam comparações entre as medianas de performance de grupos de campanhas com características distintas.

Por exemplo, uma diferença de 20% no CTR significa que o grupo com a característica recomendada apresentou CTR mediano 20% superior ao grupo atual. **Isso não significa que aplicar a alteração aumentará o CTR da campanha em 20%.**

As recomendações devem ser interpretadas como oportunidades de melhoria que podem orientar novos testes, e não como previsões de ganhos garantidos.

O protótipo apresenta algumas limitações:

- As comparações são observacionais e não estabelecem relações causais.
- A análise considera a plataforma e o atributo individualmente, sem controlar simultaneamente todas as características das campanhas.
- As alterações são avaliadas separadamente, sem estimar o efeito conjunto de múltiplas recomendações.
- O tamanho limitado do dataset restringe a quantidade de comparações possíveis.

Para detalhes sobre a implementação, as decisões metodológicas e os testes realizados, consulte `parte3_recomendacoes.ipynb`.
